# Matplotlib & Seaborn Practice — Home Credit Default Risk

A full practice workbook covering the matplotlib and seaborn skills you need for credit risk EDA, built on `application_train.csv`.

**How to use this notebook**

1. Work through sections in order — matplotlib fundamentals first, then seaborn, then a capstone.
2. Each section gives you brief context and the key syntax, then exercises. Write your code in the empty cells.
3. Every section also has **interpretation questions** — answer them in the markdown cells provided. Saying what a plot *shows* and what it *means for credit risk* is the skill that separates an analyst from a chart-maker.
4. Bring your attempts to Claude for marking, or ask for hints when stuck. Try each exercise cold first.

**Ground rules**

- Every plot gets a title and axis labels. No exceptions — build the habit now.
- Before running each plot, predict what it will look like (hypothesis-before-analysis).
- If a plot surprises you, that is a finding. Write it down.

## 0 — Setup and data loading

Run these cells to get started. The load cell follows your usual repo-relative pattern — adjust `DATA_PATH` if your data lives elsewhere.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 130)
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn:", sns.__version__)

In [ ]:
from pathlib import Path

def find_repo_root(marker=".git"):
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    return p

REPO_ROOT = find_repo_root()
# Adjust this to wherever application_train.csv sits in your repo:
DATA_PATH = REPO_ROOT / "Data" / "home-credit-default-risk" / "application_train.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)

### Derived columns

The `DAYS_*` columns are negative day counts relative to the application date, which makes for unreadable axes. Create clean versions once here and use them throughout.

**Exercise 0.1** — Create the following columns:

- `AGE_YEARS` — age in years from `DAYS_BIRTH` (positive, in years)
- `YEARS_EMPLOYED` — from `DAYS_EMPLOYED` (positive, in years). Do **not** clean the anomaly yet — a later exercise depends on it being dirty.
- `CREDIT_INCOME_RATIO` — `AMT_CREDIT / AMT_INCOME_TOTAL`
- `ANNUITY_INCOME_RATIO` — `AMT_ANNUITY / AMT_INCOME_TOTAL`

Then print `df[["AGE_YEARS", "YEARS_EMPLOYED"]].describe()` and look at the max of `YEARS_EMPLOYED`.

**Interpretation 0.1** — What do you notice about the maximum of `YEARS_EMPLOYED`? What do you suspect it encodes, and why would a lender's system produce a value like that?

In [ ]:
# Ex 0.1 — derived columns

**Your answer (0.1):**

...

## 1 — Anatomy of a figure

Matplotlib has two APIs: the quick **pyplot** interface (`plt.plot(...)`) and the **object-oriented (OO)** interface (`fig, ax = plt.subplots()` then `ax.plot(...)`). The OO interface is the one to build muscle memory on — it scales to multi-panel figures and is what serious EDA code uses.

The object hierarchy:

- **Figure** — the whole canvas. Owns one or more Axes.
- **Axes** — one plot (the thing with an x-axis and y-axis). This is where 95% of your method calls go.
- **Axis** — the x or y axis object itself (ticks, limits, labels).
- **Artist** — everything drawn (lines, patches, text) is an Artist.

**Key syntax**

```python
fig, ax = plt.subplots(figsize=(8, 5))   # one Figure, one Axes
ax.plot(x, y)                            # draw on the Axes
ax.set_title("Title")
ax.set_xlabel("x label")
ax.set_ylabel("y label")
plt.show()
```

### Exercises

**1.1** — Using the OO interface, make a simple line plot of `y = x**2` for `x = np.arange(0, 10)`. Title it and label both axes. (Yes, it is a toy — the point is the API pattern.)

**1.2** — Same plot, but written with the pyplot interface (`plt.plot`, `plt.title`, ...). Confirm you get the same output.

**Interpretation 1.1** — In your own words: what is the difference between the Figure and the Axes? Why does the OO interface matter once you start building 2×2 comparison panels for an EDA report?

**Interpretation 1.2** — When you see `plt.gca()` in someone else's code, what is it doing?

In [ ]:
# Ex 1.1 — OO interface

In [ ]:
# Ex 1.2 — pyplot interface

**Your answers (1.1, 1.2):**

...

## 2 — Line plots and basic styling

Line plots show how a value changes across an ordered variable. In credit risk you rarely plot raw rows as lines — you plot **aggregates**: default rate by age band, approval volume by month, rate by score decile.

**Key syntax**

```python
ax.plot(x, y,
        color="tab:blue",      # named colours, hex, or "C0".."C9"
        linestyle="--",        # "-", "--", ":", "-."
        linewidth=2,
        marker="o",            # add point markers
        label="series name")   # picked up by ax.legend()
ax.legend()
```

### Exercises

**2.1** — Compute the mean of `TARGET` grouped by `HOUR_APPR_PROCESS_START` (this is the default rate by hour the application was started). Plot it as a line with circle markers. Title, labels, and a legend entry.

**2.2** — Add a second line to the same Axes: the **number of applications** per hour. Wait — that has a totally different scale. For now, plot it anyway and observe the problem. (Section 8 fixes this properly with a twin axis.)

**2.3** — Restyle 2.1: dashed red line, linewidth 2, square markers. Purely a syntax rep.

**Interpretation 2.1** — Describe the default-rate-by-hour pattern. What might explain applications started at unusual hours having different risk? Would you trust the hours with very few applications?

**Interpretation 2.2** — Why is plotting two very different scales on one axis misleading? What are the two standard fixes?

In [ ]:
# Ex 2.1 — default rate by application hour

In [ ]:
# Ex 2.2 — add application volume (observe the scale problem)

In [ ]:
# Ex 2.3 — restyle

**Your answers (2.1, 2.2):**

...

## 3 — Histograms

The workhorse of univariate EDA. In credit risk you histogram incomes, loan amounts, ages, and scores to find skew, outliers, data-entry artefacts, and sentinel values before they poison a model.

**Key syntax**

```python
ax.hist(series.dropna(), bins=50,
        edgecolor="white",     # visible bin borders
        alpha=0.7,
        density=True)          # normalise to a density instead of counts
ax.hist(a, bins=bins, alpha=0.5, label="A")   # overlay two with shared bins
bins = np.linspace(lo, hi, 41)                # explicit shared bin edges
```

### Exercises

**3.1** — Histogram of `AGE_YEARS` with 40 bins. Then re-run with 5 bins and 200 bins. Keep all three (separate cells or a loop).

**3.2** — Histogram of `AMT_INCOME_TOTAL`. It will look terrible. Diagnose why, then fix it two ways: (a) clip/filter to below the 99th percentile, (b) keep all data but pass `np.log10` of income. Plot both.

**3.3** — Overlay the `AGE_YEARS` distributions for defaulters (`TARGET == 1`) and non-defaulters (`TARGET == 0`) on one Axes. Use `density=True`, shared explicit bins, `alpha=0.5`, and a legend. This is the single most important histogram pattern in this dataset — you will reuse it constantly.

**3.4** — Histogram of `YEARS_EMPLOYED` (still dirty). Find the anomaly visually, then create `YEARS_EMPLOYED_CLEAN` where the sentinel is replaced with `np.nan`, and re-plot.

**Interpretation 3.1** — What did 5 bins hide and what did 200 bins invent? How do you choose?

**Interpretation 3.2** — Why is `density=True` essential in 3.3 when the two groups have very different sizes? What would raw counts mislead you into thinking?

**Interpretation 3.3** — From 3.3: which age groups default more? Give a plausible economic explanation, and one policy implication a lender might (carefully, legally) consider.

**Interpretation 3.4** — What does the `DAYS_EMPLOYED` sentinel most likely represent? Why replace with NaN rather than 0 or the median at the EDA stage?

In [ ]:
# Ex 3.1 — AGE_YEARS histograms, three bin counts

In [ ]:
# Ex 3.2 — income histogram, then two fixes

In [ ]:
# Ex 3.3 — age distribution by TARGET (density, shared bins)

In [ ]:
# Ex 3.4 — find and clean the YEARS_EMPLOYED sentinel

**Your answers (3.1–3.4):**

...

## 4 — Bar charts

Bars compare quantities across categories: default rate by education, counts by income type. Rule of thumb — **horizontal bars** (`barh`) whenever category names are long, and **sort the bars** unless the category has a natural order.

**Key syntax**

```python
counts = df["COL"].value_counts()
rates = df.groupby("COL")["TARGET"].mean().sort_values()

ax.bar(counts.index, counts.values)
ax.barh(rates.index, rates.values)             # horizontal
ax.tick_params(axis="x", rotation=45)          # or use barh instead

# grouped bars: shift positions by width
x = np.arange(len(labels)); w = 0.4
ax.bar(x - w/2, series_a, width=w, label="A")
ax.bar(x + w/2, series_b, width=w, label="B")
ax.set_xticks(x, labels)
```

### Exercises

**4.1** — Vertical bar chart of application counts by `NAME_INCOME_TYPE`. Rotate the x labels so they are readable.

**4.2** — Horizontal bar chart of **default rate** by `NAME_EDUCATION_TYPE`, sorted ascending. Add a vertical reference line at the overall default rate (`df["TARGET"].mean()`).

**4.3** — Grouped bar chart: for `CODE_GENDER` (drop the XNA row), show two bars per gender — count of non-defaulters and count of defaulters. Legend required.

**4.4** — Default rate by `OCCUPATION_TYPE`, horizontal, sorted, with the overall-rate reference line. There are ~18 categories — this is exactly when `barh` earns its keep.

**Interpretation 4.1** — From 4.2: which education levels sit above the portfolio-average default rate? Is the pattern monotonic with education? Why might education correlate with default even though no lender prices on education directly?

**Interpretation 4.2** — In 4.3, why is a *grouped count* chart nearly useless for comparing risk between genders? What chart answers the risk question directly?

**Interpretation 4.3** — From 4.4: name the two highest-risk and two lowest-risk occupations. What income-stability story connects them?

In [ ]:
# Ex 4.1 — counts by income type

In [ ]:
# Ex 4.2 — default rate by education, sorted, with reference line

In [ ]:
# Ex 4.3 — grouped bars by gender and target

In [ ]:
# Ex 4.4 — default rate by occupation

**Your answers (4.1–4.3):**

...

## 5 — Scatter plots

Scatters show the joint distribution of two numerics. With 307k rows, naive scatters become solid ink blobs — the skill here is **overplotting management**: sampling, alpha, and colour mapping.

**Key syntax**

```python
sample = df.sample(5000, random_state=42)
ax.scatter(x, y,
           s=8,                # marker size
           alpha=0.2,          # transparency fights overplotting
           c=values,           # colour by a third variable
           cmap="viridis")
fig.colorbar(sc, ax=ax, label="third variable")   # sc = the scatter handle
```

### Exercises

**5.1** — Scatter `AMT_CREDIT` (y) vs `AMT_GOODS_PRICE` (x) on a 5,000-row sample. First with defaults, then tuned (`s=8, alpha=0.2`). Note the structure you can see.

**5.2** — Scatter `AMT_CREDIT` vs `AMT_INCOME_TOTAL` on a sample, colouring points by `TARGET` (two colours — plot the two groups as two `scatter` calls so you get a proper legend). Filter income to below the 99th percentile first.

**5.3** — Scatter `EXT_SOURCE_2` (x) vs `EXT_SOURCE_3` (y) on a sample, coloured by `AGE_YEARS` with a continuous colormap and a colorbar.

**Interpretation 5.1** — In 5.1 you should see points hugging a line, plus banding. What is the credit vs goods-price relationship telling you about how these loans are structured? What causes the banding?

**Interpretation 5.2** — Can you visually separate defaulters from non-defaulters in 5.2? What does that tell you about how useful raw income and credit amount will be as standalone model features?

**Interpretation 5.3** — The EXT_SOURCE columns are external credit scores. From 5.3, are the two scores correlated? Does age appear related to either? Why do normalised external scores so often end up as the strongest features in this competition?

In [ ]:
# Ex 5.1 — credit vs goods price

In [ ]:
# Ex 5.2 — credit vs income coloured by TARGET

In [ ]:
# Ex 5.3 — EXT_SOURCE_2 vs EXT_SOURCE_3 coloured by age

**Your answers (5.1–5.3):**

...

## 6 — Customisation: ticks, formatters, grids, spines

Default matplotlib output says "draft". Formatted ticks, a light grid, and removed top/right spines say "report". These few lines are most of the difference.

**Key syntax**

```python
ax.set_xlim(20, 70); ax.set_ylim(0, 0.12)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

# tick formatting
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))   # 0.08 -> 8%
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))  # 1,000,000
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))             # tick every 5

fig.suptitle("Figure-level title", fontsize=14)
fig.tight_layout()
```

### Exercises

**6.1** — Rebuild your 4.2 chart (default rate by education) with: percent-formatted value axis, light grid on the value axis only, top and right spines removed, and a proper title stating the takeaway (a "headline title", e.g. "Lower secondary education carries the highest default rate" rather than "Default rate by education").

**6.2** — Rebuild 3.2's clipped income histogram with thousands-separator x ticks and spines removed.

**6.3** — Take your 3.3 age-by-target overlay and set x limits to the actual age range (roughly 20–70), add `MultipleLocator(5)` x ticks, and a grid.

**Interpretation 6.1** — Why are headline titles better than descriptive titles in a report for a credit committee? Give one situation where a neutral descriptive title is the right choice instead.

In [ ]:
# Ex 6.1 — polished education chart

In [ ]:
# Ex 6.2 — polished income histogram

In [ ]:
# Ex 6.3 — polished age overlay

**Your answer (6.1):**

...

## 7 — Annotations and reference lines

Annotations turn a chart into an argument: mark the portfolio average, flag the anomaly, label the number that matters.

**Key syntax**

```python
ax.axhline(0.08, color="red", linestyle="--", label="portfolio avg")
ax.axvline(threshold, color="grey", linestyle=":")
ax.axvspan(60, 70, alpha=0.15, color="orange")        # shaded band

ax.text(x, y, "label", ha="center", va="bottom")
ax.annotate("sentinel value",
            xy=(x_point, y_point),          # arrow tip
            xytext=(x_text, y_text),        # text position
            arrowprops=dict(arrowstyle="->"))

# value labels on bars
for bar in bars:
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f" {bar.get_width():.1%}", va="center")
```

### Exercises

**7.1** — Take the 4.4 occupation chart and add a percentage value label at the end of each bar.

**7.2** — Histogram of the *raw* `DAYS_EMPLOYED` (all data, before cleaning) and use `ax.annotate` with an arrow to point at the sentinel spike, labelled with what you concluded it means.

**7.3** — On your 2.1 default-rate-by-hour line, shade the band of hours you consider "out of business hours" with `axvspan`, and add an `axhline` at the overall default rate.

**Interpretation 7.1** — Annotation is persuasive. What is the risk of over-annotating, or of annotating only the points that support your prior? How would you keep yourself honest?

In [ ]:
# Ex 7.1 — bar value labels

In [ ]:
# Ex 7.2 — annotate the sentinel

In [ ]:
# Ex 7.3 — shaded band + reference line

**Your answer (7.1):**

...

## 8 — Log scales and twin axes

Money variables are right-skewed almost by definition — a handful of very large incomes crush the rest of the axis. Log scales fix that. Twin axes solve the two-different-units problem from Exercise 2.2 (use sparingly; they are easy to abuse).

**Key syntax**

```python
ax.set_xscale("log")          # log the axis (keeps original units on ticks)
ax.hist(np.log10(series))     # vs transforming the data itself

ax2 = ax.twinx()              # second y-axis sharing the same x
ax2.plot(x, y2, color="tab:orange")
ax2.set_ylabel("second unit")
```

### Exercises

**8.1** — Histogram of `AMT_INCOME_TOTAL` (all rows, no clipping) with `ax.set_xscale("log")`. Compare mentally with your 3.2 approaches.

**8.2** — Fix Exercise 2.2 properly: default rate by hour as a line on the left axis (percent-formatted), application volume per hour as bars on a twin right axis (drawn first, in a light colour, so the line stays readable). Label both axes clearly.

**8.3** — Boxplot-free skew check: plot `.plot(kind="hist")`-style histograms of `CREDIT_INCOME_RATIO` on linear and log x scales side by side in one figure (peek at Section 9 for `plt.subplots(1, 2)`).

**Interpretation 8.1** — When presenting to a non-technical credit committee, what is the danger of a log axis? How would you caption it?

**Interpretation 8.2** — In 8.2, which hours have high default rates but tiny volume? Why should the volume context change how much you trust — and how you would act on — the rate line?

In [ ]:
# Ex 8.1 — log-scale income

In [ ]:
# Ex 8.2 — rate line + volume bars on twin axes

In [ ]:
# Ex 8.3 — linear vs log side by side

**Your answers (8.1, 8.2):**

...

## 9 — Subplot layouts and saving figures

Multi-panel figures are the format of real EDA output: one figure, four related views, one takeaway. This is where the OO interface pays off.

**Key syntax**

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(...)                   # index the 2D array of Axes
for ax, col in zip(axes.flat, cols):   # or iterate over axes.flat
    ax.hist(df[col].dropna(), bins=40)
    ax.set_title(col)

fig, axes = plt.subplots(1, 2, sharey=True)   # shared axis limits
fig.suptitle("One overall title")
fig.tight_layout()

fig.savefig("figure.png", dpi=150, bbox_inches="tight")
```

### Exercises

**9.1** — A 2×2 grid of histograms for `AMT_INCOME_TOTAL` (clipped), `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`. Loop over `axes.flat` rather than writing four blocks. Each panel titled; one `suptitle`.

**9.2** — A 1×3 grid: distributions of `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, each split by TARGET (your 3.3 overlay pattern), with `sharey=True`.

**9.3** — Save the 9.2 figure as a PNG at `dpi=150` with `bbox_inches="tight"`, then confirm the file exists.

**Interpretation 9.1** — From 9.2: rank the three external sources by how well they visually separate defaulters from non-defaulters. What does "separation" in a density overlay correspond to in model terms?

**Interpretation 9.2** — Why `sharey=True` when comparing the three panels? What comparison would be corrupted without it?

In [ ]:
# Ex 9.1 — 2x2 money histograms

In [ ]:
# Ex 9.2 — EXT_SOURCE panels by TARGET

In [ ]:
# Ex 9.3 — savefig

**Your answers (9.1, 9.2):**

...

## 10 — Seaborn: how it relates to matplotlib

Seaborn is a layer on top of matplotlib: it takes a DataFrame plus column names, handles grouping/aggregation/legends for you, and draws using matplotlib underneath. Everything you learned in Sections 1–9 still applies — most seaborn functions return or draw onto a matplotlib `Axes`, so you polish seaborn plots with the same `ax.set_*` calls.

The one distinction that prevents 90% of seaborn confusion:

- **Axes-level functions** (`sns.histplot`, `sns.boxplot`, `sns.scatterplot`, `sns.heatmap`, ...) draw on a single Axes. They accept `ax=` and slot into `plt.subplots` grids.
- **Figure-level functions** (`sns.displot`, `sns.catplot`, `sns.relplot`, `sns.lmplot`, `sns.jointplot`, `sns.pairplot`) create their **own figure** (a `FacetGrid`). They do NOT accept `ax=` — you cannot put them inside your own subplot grid, but they can facet into columns/rows on their own.

**Key syntax**

```python
sns.set_theme(style="whitegrid")           # opt in to seaborn styling

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x="AGE_YEARS", ax=ax)   # axes-level: composable
ax.set_title("still plain matplotlib methods")

sns.displot(data=df, x="AGE_YEARS", col="TARGET")   # figure-level: own figure
```

### Exercises

**10.1** — Call `sns.set_theme(style="whitegrid")`. Re-run one of your Section 3 histograms unchanged and note what changed visually.

**10.2** — Draw `sns.histplot(data=df, x="AGE_YEARS")` into an Axes you created with `plt.subplots`, and set the title with matplotlib. Then try passing `ax=` to `sns.displot` and read the warning/error you get.

**Interpretation 10.1** — In your own words: when would you reach for the axes-level function and when for the figure-level one? Which will you use inside a 2×2 EDA panel?

In [ ]:
# Ex 10.1 — set_theme, re-run an old plot

In [ ]:
# Ex 10.2 — axes-level vs figure-level

**Your answer (10.1):**

...

## 11 — Distribution plots: histplot, kdeplot, ecdfplot

Seaborn's distribution tools do in one line what took you several in Section 3 — especially splitting by a grouping variable with `hue`. In this dataset, `hue="TARGET"` is the move you will make hundreds of times.

**Key syntax**

```python
sns.histplot(data=df, x="COL", hue="TARGET",
             stat="density",        # like density=True
             common_norm=False,     # CRITICAL: normalise each group separately
             element="step",        # cleaner overlays than bars
             bins=50, ax=ax)

sns.kdeplot(data=df, x="COL", hue="TARGET", common_norm=False,
            fill=True, alpha=0.3, ax=ax)

sns.ecdfplot(data=df, x="COL", hue="TARGET", ax=ax)
```

### Exercises

**11.1** — Redo your 3.3 age-by-target overlay in ONE `sns.histplot` call using `hue`, `stat="density"`, `common_norm=False`, `element="step"`. Compare effort with your matplotlib version.

**11.2** — Run the same plot but with `common_norm=True` (the default). Observe how the defaulter distribution almost vanishes.

**11.3** — `sns.kdeplot` of `EXT_SOURCE_3` by TARGET, filled. Then the same variable as `sns.ecdfplot` by TARGET.

**11.4** — KDE of `AMT_CREDIT` by TARGET with `log_scale=True`. (Yes, `log_scale` is built in.)

**Interpretation 11.1** — Explain precisely what `common_norm=False` does and why it is essential with an ~8%/92% class imbalance. What lie does `common_norm=True` tell here?

**Interpretation 11.2** — KDE vs histogram: what does the KDE smooth over, and when is that smoothing dangerous? (Hint: think about your sentinel spike and hard boundaries like age 21.)

**Interpretation 11.3** — Read the ECDF from 11.3: roughly what fraction of *defaulters* have `EXT_SOURCE_3` below 0.4, vs non-defaulters? Why are ECDFs the honest choice for comparing groups (no bins, no bandwidth)?

In [ ]:
# Ex 11.1 — histplot with hue

In [ ]:
# Ex 11.2 — common_norm=True (observe the problem)

In [ ]:
# Ex 11.3 — kdeplot and ecdfplot of EXT_SOURCE_3

In [ ]:
# Ex 11.4 — log-scale KDE of credit amount

**Your answers (11.1–11.3):**

...

## 12 — Categorical plots: countplot, barplot, boxplot, violinplot, stripplot

The categorical family answers "how does a numeric differ across groups" and "how big is each group". Know what each estimates:

- `countplot` — row counts per category (a `value_counts` bar chart)
- `barplot` — the **mean** of y per category, with a bootstrap confidence interval. With a 0/1 target, the mean IS the default rate — so `sns.barplot(x=cat, y="TARGET")` gives default rate with error bars for free.
- `boxplot` — median, quartiles, whiskers, outliers
- `violinplot` — full KDE shape per group
- `stripplot`/`swarmplot` — raw points (sample first!)

**Key syntax**

```python
sns.countplot(data=df, y="NAME_INCOME_TYPE",
              order=df["NAME_INCOME_TYPE"].value_counts().index, ax=ax)

sns.barplot(data=df, x="NAME_EDUCATION_TYPE", y="TARGET",
            order=[...], errorbar=("ci", 95), ax=ax)   # mean of 0/1 = default rate

sns.boxplot(data=df, x="TARGET", y="AGE_YEARS", ax=ax)
sns.violinplot(data=df, x="TARGET", y="EXT_SOURCE_2", ax=ax)
sns.violinplot(..., hue="CODE_GENDER", split=True)     # split violins
```

### Exercises

**12.1** — `countplot` of `NAME_FAMILY_STATUS`, horizontal, ordered by frequency.

**12.2** — `barplot` of default rate by `NAME_EDUCATION_TYPE` with 95% CIs, ordered by rate. Compare with your hand-built 4.2: what did you get for free?

**12.3** — `boxplot` of `AMT_INCOME_TOTAL` by `TARGET`. Diagnose why it is unreadable, then fix (filter extreme incomes or log scale).

**12.4** — `violinplot` of `EXT_SOURCE_2` by `TARGET`. Then a split violin: `x="TARGET"`, `y="AGE_YEARS"`, `hue="CODE_GENDER"` (drop XNA), `split=True`.

**12.5** — `stripplot` of `AMT_ANNUITY` by `NAME_CONTRACT_TYPE` on a 2,000-row sample with `alpha=0.3`. Optionally overlay a boxplot on the same Axes.

**12.6** — Default rate by `REGION_RATING_CLIENT` as a barplot. Note this category is ordinal (1, 2, 3) — keep the natural order.

**Interpretation 12.1** — In 12.2, the CI whiskers are wide for some education levels and narrow for others. What drives the width, and how should it affect your confidence in the rate ranking?

**Interpretation 12.2** — Boxes vs violins: what did the violin in 12.4 reveal about `EXT_SOURCE_2` that the five-number summary of a boxplot compresses away?

**Interpretation 12.3** — Does `REGION_RATING_CLIENT` behave monotonically with default? Why does monotonicity matter if you later want to use a feature in a scorecard-style (logistic) model?

**Interpretation 12.4** — Median incomes for defaulters vs non-defaulters are close. Why can a variable with near-identical group medians still be useful in a model?

In [ ]:
# Ex 12.1 — countplot family status

In [ ]:
# Ex 12.2 — barplot default rate by education with CIs

In [ ]:
# Ex 12.3 — income boxplot, diagnose and fix

In [ ]:
# Ex 12.4 — violins and split violins

In [ ]:
# Ex 12.5 — stripplot on a sample

In [ ]:
# Ex 12.6 — default rate by region rating

**Your answers (12.1–12.4):**

...

## 13 — Relational and regression plots: scatterplot, regplot, lmplot, jointplot

Seaborn's `scatterplot` adds `hue`/`size`/`style` semantics to what you built manually in Section 5. `regplot` overlays a fitted regression line; `jointplot` glues a scatter to marginal distributions.

**Key syntax**

```python
sns.scatterplot(data=sample, x="A", y="B",
                hue="TARGET", alpha=0.3, s=15, ax=ax)

sns.regplot(data=sample, x="A", y="B",
            scatter_kws=dict(alpha=0.2, s=10),
            line_kws=dict(color="red"), ax=ax)

sns.jointplot(data=sample, x="A", y="B", kind="scatter")   # figure-level
# kind: "scatter", "hex", "kde", "hist", "reg"

sns.lmplot(data=sample, x="A", y="B", hue="TARGET")        # figure-level regplot
```

### Exercises

**13.1** — Redo 5.2 (credit vs income, coloured by TARGET) as a single `sns.scatterplot` with `hue`. One line plus polish.

**13.2** — `regplot` of `AMT_ANNUITY` vs `AMT_CREDIT` on a 5,000-row sample. Style the scatter faint and the line red.

**13.3** — `jointplot` of `EXT_SOURCE_2` vs `EXT_SOURCE_3` on a sample, `kind="hex"`. Try `kind="kde"` too.

**13.4** — `lmplot` of `AMT_ANNUITY` vs `AMT_CREDIT` with `hue="NAME_CONTRACT_TYPE"` on a sample. Two fitted lines.

**Interpretation 13.1** — The annuity–credit relationship in 13.2 is nearly deterministic. What does an annuity actually represent (you know this from mortgage lending), and why is `AMT_ANNUITY / AMT_CREDIT` roughly the same for loans of the same term?

**Interpretation 13.2** — In 13.4, cash loans and revolving loans show different annuity-per-credit slopes. What does that reflect about the two products?

**Interpretation 13.3** — When is fitting a straight line (`regplot`) actively misleading? Name a check you would run before trusting it.

In [ ]:
# Ex 13.1 — scatterplot with hue

In [ ]:
# Ex 13.2 — regplot annuity vs credit

In [ ]:
# Ex 13.3 — jointplot hex / kde

In [ ]:
# Ex 13.4 — lmplot by contract type

**Your answers (13.1–13.3):**

...

## 14 — Heatmaps: correlations, crosstabs, missingness

`sns.heatmap` renders any 2-D matrix as colour. In credit EDA the big three uses are correlation matrices, category×category rate tables, and missingness patterns.

**Key syntax**

```python
corr = df[num_cols].corr()
sns.heatmap(corr,
            annot=True, fmt=".2f",       # print values in the cells
            cmap="coolwarm",
            vmin=-1, vmax=1, center=0,   # anchor the diverging colormap
            square=True, ax=ax)

mask = np.triu(np.ones_like(corr, dtype=bool))   # hide the duplicate triangle
sns.heatmap(corr, mask=mask, ...)

pivot = df.pivot_table(index="CAT_A", columns="CAT_B",
                       values="TARGET", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".1%", cmap="Reds", ax=ax)
```

### Exercises

**14.1** — Correlation heatmap of: `TARGET`, `AGE_YEARS`, `YEARS_EMPLOYED_CLEAN`, `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, `CREDIT_INCOME_RATIO`. Annotated, `coolwarm`, centred at 0, upper triangle masked.

**14.2** — Default-rate pivot heatmap: `NAME_EDUCATION_TYPE` (rows) × `NAME_FAMILY_STATUS` (columns), values = mean TARGET, annotated as percentages, `cmap="Reds"`. Also produce the corresponding **count** pivot so you know which cells are thin.

**14.3** — Missingness heatmap: take ~20 columns spanning low and high missingness (include the EXT_SOURCEs and some building columns like `EXT_SOURCE_1`, `OWN_CAR_AGE`, `OCCUPATION_TYPE`, and a few `*_AVG` columns), build `df[cols].isna()`, and heatmap a 2,000-row sample of it (`cbar=False`, no annot).

**Interpretation 14.1** — From 14.1: which feature has the strongest (absolute) correlation with TARGET, and is it positive or negative? Why are all the target correlations small in magnitude, and why does that NOT mean the features are useless?

**Interpretation 14.2** — `AMT_CREDIT`, `AMT_ANNUITY`, and `AMT_GOODS_PRICE` form a highly correlated block. What problem does that cause for a logistic regression, and name two ways to deal with it.

**Interpretation 14.3** — From 14.2 + the count pivot: find one cell with an extreme rate. Is it trustworthy? What minimum-count rule would you apply before quoting cell-level rates to stakeholders?

**Interpretation 14.4** — From 14.3: does missingness look random, or do columns go missing together in blocks? What does block-missingness usually indicate about how the data was collected, and how does it change your imputation strategy?

In [ ]:
# Ex 14.1 — correlation heatmap with mask

In [ ]:
# Ex 14.2 — education x family status default-rate pivot (+ counts)

In [ ]:
# Ex 14.3 — missingness heatmap

**Your answers (14.1–14.4):**

...

## 15 — Faceting: catplot, displot, relplot, FacetGrid, pairplot

Faceting = same plot, repeated across subsets ("small multiples"). The figure-level functions (`catplot`, `displot`, `relplot`) are the front doors; `FacetGrid` is the engine underneath; `pairplot` is a grid of pairwise scatters.

**Key syntax**

```python
sns.displot(data=df, x="AGE_YEARS", hue="TARGET",
            col="NAME_CONTRACT_TYPE",      # one column of panels per category
            stat="density", common_norm=False, element="step")

sns.catplot(data=df, x="TARGET", y="EXT_SOURCE_2",
            col="CODE_GENDER", kind="violin")

sns.relplot(data=sample, x="AMT_GOODS_PRICE", y="AMT_CREDIT",
            col="NAME_INCOME_TYPE", col_wrap=3, alpha=0.3)

sns.pairplot(sample[cols + ["TARGET"]], hue="TARGET",
             corner=True, plot_kws=dict(alpha=0.3, s=10))
```

### Exercises

**15.1** — `displot`: age distribution by TARGET (hue), faceted into columns by `NAME_CONTRACT_TYPE`.

**15.2** — `catplot`: violins of `EXT_SOURCE_2` by TARGET, faceted by `CODE_GENDER` (drop XNA first).

**15.3** — `relplot`: credit vs goods price scatter on a sample, faceted by `NAME_INCOME_TYPE` with `col_wrap=3`.

**15.4** — `pairplot` of the three EXT_SOURCE columns + `AGE_YEARS` on a 3,000-row sample, `hue="TARGET"`, `corner=True`. This is the money plot for this dataset.

**Interpretation 15.1** — From 15.1: does the age–risk relationship look the same for cash loans and revolving loans? If a relationship changes across facets, what does that imply for modelling (name the term)?

**Interpretation 15.2** — From 15.4: which single variable, or pair of variables, best separates the classes? Where in the grid do you look to answer "would a linear boundary work"?

**Interpretation 15.3** — Faceting into small subsets shrinks the data per panel. What is the trade-off, and when would you facet vs use hue on one Axes?

In [ ]:
# Ex 15.1 — displot facets

In [ ]:
# Ex 15.2 — catplot violins by gender

In [ ]:
# Ex 15.3 — relplot col_wrap

In [ ]:
# Ex 15.4 — pairplot of EXT_SOURCEs + age

**Your answers (15.1–15.3):**

...

## 16 — Themes, palettes, contexts

Three independent dials:

- **style** — background/grid look: `whitegrid`, `darkgrid`, `ticks`, `white`
- **palette** — colours. The type must match the data type:
  - *qualitative* (categories): `"colorblind"`, `"tab10"`, `"Set2"`
  - *sequential* (low→high): `"viridis"`, `"Blues"`, `"rocket"`
  - *diverging* (negative↔positive around a midpoint): `"coolwarm"`, `"vlag"`, `"RdBu"`
- **context** — scaling for the medium: `paper`, `notebook` (default), `talk`, `poster`

**Key syntax**

```python
sns.set_theme(style="ticks", palette="colorblind", context="talk")
sns.color_palette("viridis", 8)          # inspect a palette in Jupyter
sns.histplot(..., palette="Set2")        # per-plot override (with hue)
sns.despine()                            # remove top/right spines
```

### Exercises

**16.1** — Display three palettes in-notebook with `sns.color_palette(...)`: one qualitative, one sequential, one diverging.

**16.2** — Re-render your 14.1 correlation heatmap with `cmap="viridis"`. It will look plausible and be wrong. Then switch back to a diverging map centred at 0.

**16.3** — Take one favourite plot from earlier and render it twice: `context="paper"` vs `context="talk"`.

**Interpretation 16.1** — Explain why a sequential colormap on a correlation matrix is a genuine error, not a style preference. What question can a reader not answer with viridis on [-1, 1]?

**Interpretation 16.2** — Why default to colorblind-safe palettes for anything a credit committee will see?

In [ ]:
# Ex 16.1 — three palettes

In [ ]:
# Ex 16.2 — wrong then right heatmap colormap

In [ ]:
# Ex 16.3 — paper vs talk context

**Your answers (16.1, 16.2):**

...

## 17 — Capstone: a one-figure risk dashboard

Everything above, combined. Build **one figure** you could drop into a portfolio README or hand to a credit manager. No new syntax — this is composition, polish, and judgement.

**The brief**

Build a 2×2 figure, `figsize=(14, 10)`, titled with a headline finding, containing:

1. **Top-left** — default rate by `NAME_EDUCATION_TYPE`: horizontal bars, sorted, percent axis, portfolio-average reference line, value labels.
2. **Top-right** — `EXT_SOURCE_3` density by TARGET (`histplot`, `hue`, `stat="density"`, `common_norm=False`, `element="step"`).
3. **Bottom-left** — default rate by 5-year age band: compute with `pd.cut` on `AGE_YEARS`, then plot as a line with markers, percent axis. (Bonus: add faint volume bars on a twin axis.)
4. **Bottom-right** — the 14.1 correlation heatmap (or a trimmed version), masked upper triangle.

Requirements: consistent styling across panels (`set_theme` once), every panel titled, spines handled, `tight_layout` or `constrained_layout`, saved to PNG at 150 dpi.

**Then write the analysis** (markdown cell below): four decision statements, one per panel, in your agreed format — *observation → decision*. Example shape: "Default rate falls monotonically with age from ~12% (20–25) to ~5% (60+) → retain AGE_YEARS as a model feature; consider banded version for scorecard interpretability."

In [ ]:
# Capstone — build the dashboard here
# (feel free to develop each panel in its own scratch cell first, then assemble)

**Capstone decision statements:**

Panel 1 (education):

Panel 2 (EXT_SOURCE_3):

Panel 3 (age bands):

Panel 4 (correlations):

## 18 — Self-test: can you answer these cold?

Close the notebook docs. If any of these are shaky, redo the relevant section.

1. Figure vs Axes — one sentence each.
2. The three lines that turn a draft chart into a report chart.
3. When you must use `common_norm=False`, and what goes wrong without it.
4. Axes-level vs figure-level seaborn functions — and which one accepts `ax=`.
5. What `sns.barplot(x=cat, y="TARGET")` estimates, and what the whiskers are.
6. Boxplot vs violin — what each shows and hides.
7. Why a diverging colormap for correlations and a sequential one for rates.
8. Three ways to handle overplotting in a 300k-row scatter.
9. Why you check group sizes before quoting subgroup default rates.
10. The `DAYS_EMPLOYED` sentinel: what it is, how you found it, what you did about it.

**When you're done:** bring your completed sections to Claude for marking — plots and written answers both. Then this feeds straight into finishing the Home Credit EDA with decision statements.